# CDaR Optuna Parameter Calibration — Analysis

CDaR-specific counterpart of `optuna_parameter_calibration_analysis.ipynb`.

This notebook reads `scripts/configs/optuna_calibration_cdar.yaml`, preserves its CDaR-specific risk weights, and analyzes the historical calibration study in `results/calibration/bayessian_cdar_risk_control/`.

It focuses on parameters that are unique or especially important to the CDaR arm: `cdar_confidence_level`, `cdar_budget_mult`, `cdar_target_multiplier`, turnover penalty, minimum overlay, forward weight, cap, shrinkage and nested-selection stability controls.

In [25]:
%matplotlib inline
import json, os, sys, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:
    import yaml
except ImportError:
    yaml=None

ROOT=Path.cwd().resolve()
if ROOT.name.lower() in {"notebook","notebooks"}: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.optimization.analysis import (
    add_policy_scores, build_parameter_range_summary, choose_policy_trials,
    extract_pareto_front, policy_comparison_table,
)

CFG_PATH=ROOT/"scripts/configs/optuna_calibration_cdar.yaml"
assert CFG_PATH.exists(), f"Missing {CFG_PATH}"
CFG=yaml.safe_load(CFG_PATH.read_text()) if yaml else {}
OUT=ROOT/CFG.get("output_folder","results/calibration/bayessian_cdar_risk_control")
TRIAL_METRICS=OUT/"trial_metrics.csv"
print("Calibration config:",CFG_PATH)
print("Study output:",OUT)


Calibration config: /Users/mac/Downloads/Sequential-stochastic-portfolio-optimization-/scripts/configs/optuna_calibration_cdar.yaml
Study output: /Users/mac/Downloads/Sequential-stochastic-portfolio-optimization-/results/calibration/bayessian_cdar_risk_control


In [26]:
ROOT


PosixPath('/Users/mac/Downloads/Sequential-stochastic-portfolio-optimization-')

## 0. Optionally run the CDaR calibration

A 60-trial nested walk-forward calibration can be very expensive. The default is analysis-only.

In [27]:
RUN_CALIBRATION=False
if RUN_CALIBRATION:
    env=os.environ.copy(); env["PYTHONPATH"]=str(ROOT)
    cmd=[sys.executable,"-m","scripts.optimization.optuna_calibration","--config",str(CFG_PATH)]
    print(" ".join(cmd));
    proc=subprocess.run(cmd,cwd=str(ROOT),env=env)
    if proc.returncode!=0: raise RuntimeError(f"Calibration failed with exit={proc.returncode}")
else:
    print("RUN_CALIBRATION=False: using existing trial_metrics.csv")


RUN_CALIBRATION=False: using existing trial_metrics.csv


## 1. Load trials with the **CDaR** selection utility from the YAML

In [28]:
if not TRIAL_METRICS.exists():
    raise FileNotFoundError(f"Run calibration first; missing {TRIAL_METRICS}")
raw=pd.read_csv(TRIAL_METRICS)
sel=CFG.get("selection",{}) or {}
trials=add_policy_scores(raw, return_weights=sel.get("return_weights"), risk_weights=sel.get("risk_weights"))
completed=trials[trials.get("status",pd.Series("COMPLETE",index=trials.index)).astype(str).str.upper().eq("COMPLETE")].copy()
print("Trials:",len(raw),"completed:",len(completed))
print("Tail-risk objective:",CFG.get("tail_risk_objective"))
display(completed.head())


FileNotFoundError: Run calibration first; missing /Users/mac/Downloads/Sequential-stochastic-portfolio-optimization-/results/calibration/bayessian_cdar_risk_control/trial_metrics.csv

## 2. Select maximum-return and CDaR risk-control policies

In [ ]:
selected=choose_policy_trials(
    completed,
    return_floor_quantile=float(sel.get("return_floor_quantile",0.35)),
    minimum_annual_return=sel.get("minimum_annual_return"),
    minimum_dsr=sel.get("minimum_dsr"),
)
comparison=policy_comparison_table(completed,selected)
display(comparison)

for name,row in selected.items():
    params={c.removeprefix("param_"):row[c] for c in completed.columns if c.startswith("param_") and c in row.index}
    print("\n",name,"parameters")
    print(json.dumps(params,indent=2,default=str))


## 3. Return–CDaR and return–drawdown frontiers

In [ ]:
def mark_selected(ax,xcol,ycol):
    for name,row in selected.items():
        if xcol in row and ycol in row:
            ax.scatter([row[xcol]],[row[ycol]],s=130,marker="*",edgecolor="black",label=name)
            ax.annotate(name,(row[xcol],row[ycol]),xytext=(6,6),textcoords="offset points")

if "cdar_95" in completed:
    fig,ax=plt.subplots(figsize=(9,6.5))
    sc=ax.scatter(100*completed["cdar_95"].abs(),100*completed["annualized_return"],c=completed["risk_control_score"],cmap="viridis",alpha=.75)
    # stars in original units need separate call below
    for name,row in selected.items():
        ax.scatter(100*abs(float(row.get("cdar_95",np.nan))),100*float(row["annualized_return"]),s=150,marker="*",edgecolor="black",label=name)
    ax.set_xlabel("|CDaR 95%| (%)"); ax.set_ylabel("Annualized return (%)"); ax.set_title("CDaR calibration frontier")
    ax.grid(alpha=.25); fig.colorbar(sc,ax=ax,label="CDaR risk-control score"); ax.legend(); plt.show()

if "max_drawdown" in completed:
    fig,ax=plt.subplots(figsize=(9,6.5))
    sc=ax.scatter(100*completed["max_drawdown"].abs(),100*completed["annualized_return"],c=completed["risk_control_score"],cmap="viridis",alpha=.75)
    for name,row in selected.items(): ax.scatter(100*abs(float(row.get("max_drawdown",np.nan))),100*float(row["annualized_return"]),s=150,marker="*",edgecolor="black",label=name)
    ax.set_xlabel("|Maximum drawdown| (%)"); ax.set_ylabel("Annualized return (%)"); ax.set_title("Return vs realized drawdown")
    ax.grid(alpha=.25); fig.colorbar(sc,ax=ax,label="CDaR risk-control score"); ax.legend(); plt.show()


## 4. Trial progression and convergence

In [ ]:
order="trial_number" if "trial_number" in completed else completed.index.name
x=completed["trial_number"] if "trial_number" in completed else np.arange(len(completed))
fig,axes=plt.subplots(3,1,figsize=(12,9),sharex=True)
axes[0].plot(x,100*completed["annualized_return"],marker="o",ms=3,lw=1); axes[0].set_ylabel("Ann. return %")
if "cdar_95" in completed: axes[1].plot(x,100*completed["cdar_95"].abs(),marker="o",ms=3,lw=1); axes[1].set_ylabel("|CDaR95| %")
axes[2].plot(x,completed["risk_control_score"],marker="o",ms=3,lw=1); axes[2].set_ylabel("Risk-control score"); axes[2].set_xlabel("Trial")
for a in axes: a.grid(alpha=.25)
axes[0].set_title("CDaR Optuna trial progression")
plt.show()


## 5. Parameter-range response table and top CDaR-control ranges

In [ ]:
ranges=build_parameter_range_summary(
    completed,
    max_bins=int(sel.get("parameter_bins",6)),
    top_fraction=float(sel.get("top_fraction",0.20)),
)
display(ranges.head(50))

metric="median_risk_control_score"
if metric in ranges:
    top=ranges.sort_values(metric,ascending=False).head(25)
    fig,ax=plt.subplots(figsize=(12,8))
    labels=top["parameter"]+" = "+top["range"]
    ax.barh(labels[::-1],top[metric][::-1]); ax.set_xlabel("Median CDaR risk-control score")
    ax.set_title("Most favorable parameter ranges"); ax.grid(axis="x",alpha=.25); plt.show()


## 6. Direct CDaR parameter effects

These plots answer the practical questions: deeper CDaR confidence level, tighter/looser drawdown budget, stronger forward timer, and higher turnover penalty — what do they do to realized CDaR, return, drawdown and turnover?

In [ ]:
interesting=[
    "param_optimizer.0.cdar_confidence_level",
    "param_optimizer.0.cdar_budget_mult",
    "param_optimizer.0.turnover_penalty",
    "param_smart_signals.cdar_target_multiplier",
    "param_smart_signals.min_overlay",
    "param_smart_signals.forward_weight",
    "param_optimizer.0.max_weight",
    "param_model.0.shrink_floor",
    "param_model.0.prior_strength",
    "param_model.0.mom_tilt",
]
metrics=[c for c in ["annualized_return","cdar_95","max_drawdown","sharpe_ratio","dsr","turnover","risk_control_score"] if c in completed]
for p in interesting:
    if p not in completed: continue
    tmp=completed[[p]+metrics].copy(); tmp[p]=pd.to_numeric(tmp[p],errors="ignore")
    grp=tmp.groupby(p,dropna=False)[metrics].median(numeric_only=True)
    if len(grp)<2: continue
    display(grp)
    cols=[c for c in ["annualized_return","cdar_95","max_drawdown","risk_control_score"] if c in grp]
    fig,axes=plt.subplots(1,len(cols),figsize=(4.2*len(cols),3.7))
    axes=np.atleast_1d(axes)
    for ax,c in zip(axes,cols):
        ax.plot(range(len(grp)),grp[c].values,marker="o"); ax.set_xticks(range(len(grp)),[str(x) for x in grp.index],rotation=30)
        ax.set_title(c); ax.grid(alpha=.25)
    fig.suptitle(p.removeprefix("param_"),y=1.03); plt.tight_layout(); plt.show()


## 7. CDaR alpha × budget interaction heatmaps

In [ ]:
a="param_optimizer.0.cdar_confidence_level"; b="param_optimizer.0.cdar_budget_mult"
if a in completed and b in completed:
    for metric in [c for c in ["cdar_95","annualized_return","risk_control_score","max_drawdown"] if c in completed]:
        P=completed.pivot_table(index=a,columns=b,values=metric,aggfunc="median")
        fig,ax=plt.subplots(figsize=(9,4.8)); im=ax.imshow(P.to_numpy(),aspect="auto",cmap="viridis")
        ax.set_xticks(range(len(P.columns)),[str(x) for x in P.columns],rotation=45); ax.set_yticks(range(len(P.index)),[str(x) for x in P.index])
        ax.set_xlabel("CDaR budget multiplier"); ax.set_ylabel("CDaR confidence alpha"); ax.set_title(f"Median {metric}: alpha × budget")
        for i in range(len(P)):
            for j in range(len(P.columns)):
                v=P.iloc[i,j]
                if pd.notna(v): ax.text(j,i,f"{v:.3g}",ha="center",va="center",fontsize=7)
        fig.colorbar(im,ax=ax,label=metric); plt.show()


## 8. Pareto-efficient CDaR configurations

In [ ]:
pareto=extract_pareto_front(completed)
cols=[c for c in ["trial_number","annualized_return","annualized_volatility","max_drawdown","cvar_95","cdar_95","sharpe_ratio","dsr","risk_control_score"] if c in pareto]
display(pareto[cols].head(30))


## 9. Parameter importance artifacts produced by the calibration runner

In [ ]:
imp_files=sorted(OUT.glob("*importance*.csv"))+sorted(OUT.glob("**/*importance*.csv"))
imp_files=list(dict.fromkeys(imp_files))
if not imp_files:
    print("No parameter-importance CSV found yet.")
else:
    for p in imp_files:
        print("\n",p.relative_to(ROOT)); display(pd.read_csv(p).head(40))


## 10. Generate YAMLs for the two selected policies

This does not run them. It writes reproducible configs using the exact selected Optuna parameter values so that you can run the maximum-return and risk-control selections through `scripts/runs/run.py` as frozen follow-up experiments.

In [ ]:
WRITE_SELECTED_YAMLS=False
if WRITE_SELECTED_YAMLS:
    if yaml is None: raise ImportError("pyyaml is required")
    from copy import deepcopy
    from scripts.optimization.optuna_calibration import set_dotted_value, sync_nested_optimizer_grid
    base_path=ROOT/CFG["base_config"]
    base=yaml.safe_load(base_path.read_text())
    gen=ROOT/"scripts/configs/generated"; gen.mkdir(parents=True,exist_ok=True)
    for policy,row in selected.items():
        cfg=deepcopy(base); changed={}
        for c in completed.columns:
            if c.startswith("param_") and c in row.index and pd.notna(row[c]):
                key=c.removeprefix("param_"); val=row[c]
                if isinstance(val,np.generic): val=val.item()
                set_dotted_value(cfg,key,val); changed[key]=val
        sync_nested_optimizer_grid(cfg,changed)
        cfg["output_folder"]=f"bayessian_cdar_optuna_{policy}"
        target=gen/f"bayessian_cdar_optuna_{policy}.yaml"
        target.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding="utf-8")
        print("wrote",target)
else:
    print("Set WRITE_SELECTED_YAMLS=True when you want frozen selected configs written.")


## 11. Export calibration notebook tables

In [ ]:
NOTE_OUT=OUT/"notebook_analysis"; NOTE_OUT.mkdir(parents=True,exist_ok=True)
comparison.to_csv(NOTE_OUT/"selected_policy_comparison.csv",index=False)
ranges.to_csv(NOTE_OUT/"parameter_range_summary.csv",index=False)
pareto.to_csv(NOTE_OUT/"pareto_front.csv",index=False)
print("Saved analysis tables to",NOTE_OUT)
